In [ ]:
"""
Week 2 - Day 2
Full DQN Training
==================
Training Deep Q-Network agent
and comparing with baselines.

Internship Spec:
"implement experience replay to
stabilize training"
"handle exploration-exploitation
trade-off using epsilon-greedy"

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')
import os
import sys

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

SRC_DIR = os.path.join(PROJECT_ROOT, "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Current Working Directory:", os.getcwd())
print("Project Root:", PROJECT_ROOT)
print("SRC Directory:", SRC_DIR)
print("SRC exists:", os.path.exists(SRC_DIR))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.dqn.dqn_agent import DQNAgent
from agents.baseline_agents import (
    FixedPriceAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)
from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)
from training.dqn_trainer import (
    plot_dqn_training,
    compare_dqn_vs_qlearning
)
from utils.price_trajectory_analyzer import (
    analyze_dqn_behavior,
    plot_trajectory_analysis
)
from utils.evaluator import evaluate_agent
from config import DQN

plt.style.use('seaborn-v0_8')
print("✅ DQN Training modules loaded!")
print(f"\nDQN Config:")
for k, v in DQN.items():
    print(f"  {k:<20}: {v}")

In [ ]:
env   = DynamicPricingEnv()
agent = DQNAgent(env, DQN)

print("Training DQN agent...")
print("2000 episodes — takes ~3-5 minutes\n")

rewards = agent.train(
    n_episodes=2000,
    verbose=True
)

print(f"\n✅ Training complete!")
print(f"   Episodes     : 2000")
print(f"   Buffer size  : {len(agent.buffer)}")
print(f"   Final ε      : {agent.epsilon:.4f}")
print(f"   Final mean   : "
      f"${np.mean(rewards[-100:]):.0f}")

In [ ]:
plot_dqn_training(
    agent,
    save_path='../results/dqn_training.png'
)

In [ ]:
print("Evaluating DQN agent (100 episodes)...")
dqn_eval = agent.evaluate(
    n_episodes=100, seed=42
)

print(f"\n  DQN Results:")
print(f"  Mean Revenue : "
      f"${dqn_eval['mean_revenue']:.0f}")
print(f"  Std Revenue  : "
      f"±${dqn_eval['std_revenue']:.0f}")
print(f"  Max Revenue  : "
      f"${dqn_eval['max_revenue']:.0f}")
print(f"  Mean Sold    : "
      f"{dqn_eval['mean_sold']:.1f}/50")

In [ ]:
# Evaluate all agents
print("Evaluating all agents...\n")

baselines = {
    'Fixed Price'  : FixedPriceAgent(env),
    'Time Based'   : TimedPricingAgent(env),
    'Demand Based' : DemandBasedAgent(env),
    'Linear Decay' : LinearDecayAgent(env)
}

all_results = {}
for name, b_agent in baselines.items():
    df = evaluate_agent(b_agent, n_episodes=100)
    all_results[name] = df['total_revenue'].mean()
    print(f"  {name:<15}: ${all_results[name]:.0f}")

# Add DQN
all_results['DQN 🤖'] = dqn_eval['mean_revenue']
print(f"  {'DQN 🤖':<15}: "
      f"${all_results['DQN 🤖']:.0f}")

# Best baseline
best_bl = max(
    v for k, v in all_results.items()
    if '🤖' not in k
)
improvement = (
    dqn_eval['mean_revenue'] - best_bl
) / best_bl * 100

print(f"\n  Best Baseline : ${best_bl:.0f}")
print(f"  DQN           : "
      f"${dqn_eval['mean_revenue']:.0f}")
print(f"  Improvement   : {improvement:+.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names    = list(all_results.keys())
revenues = list(all_results.values())
colors   = [
    'gold' if '🤖' in n
    else 'steelblue'
    for n in names
]

bars = ax.bar(
    names, revenues,
    color=colors,
    edgecolor='black',
    width=0.6
)
ax.set_title(
    'DQN vs All Baseline Agents\nMean Revenue',
    fontweight='bold', fontsize=13
)
ax.set_ylabel('Mean Revenue ($)')
ax.set_xticklabels(
    names, rotation=15, fontsize=10
)
for bar, val in zip(bars, revenues):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        val + 20,
        f'${val:.0f}',
        ha='center', fontweight='bold',
        fontsize=10
    )

# Add improvement annotation
dqn_idx = names.index('DQN 🤖')
ax.annotate(
    f'+{improvement:.1f}% vs best baseline!',
    xy=(dqn_idx, dqn_eval['mean_revenue']),
    xytext=(dqn_idx - 1.5,
            dqn_eval['mean_revenue'] + 200),
    fontsize=11, color='green',
    fontweight='bold',
    arrowprops=dict(
        arrowstyle='->',
        color='green'
    )
)

plt.tight_layout()
plt.savefig(
    '../results/dqn_vs_all.png',
    bbox_inches='tight', dpi=150
)
plt.show()
print("✅ Comparison chart saved!")

In [ ]:
analyze_dqn_behavior(agent, env)

In [ ]:
trajectory_agents = {
    'Fixed Price' : FixedPriceAgent(env),
    'Time Based'  : TimedPricingAgent(env),
    'DQN 🤖'      : agent
}

plot_trajectory_analysis(
    trajectory_agents,
    env,
    n_episodes=20,
    save_path='../results/trajectory_comparison.png'
)

In [ ]:
achieved = improvement > 0

print("╔══════════════════════════════════════════╗")
print("║    WEEK 2 DAY 2 — DQN TRAINING DONE!    ║")
print("╠══════════════════════════════════════════╣")
print("║  DQN CONFIGURATION:                      ║")
print(f"║  Episodes     : 2000{'':<21} ║")
print(f"║  Batch Size   : {DQN['batch_size']}"
      f"{'':<24} ║")
print(f"║  Buffer Size  : {DQN['memory_size']}"
      f"{'':<20} ║")
print(f"║  Target Update: every {DQN['target_update']}"
      f" eps{'':<18} ║")
print("╠══════════════════════════════════════════╣")
print("║  RESULTS:                                ║")
print(f"║  DQN Revenue  : "
      f"${dqn_eval['mean_revenue']:.0f}"
      f"{'':<22} ║")
print(f"║  Best Baseline: ${best_bl:.0f}"
      f"{'':<22} ║")
print(f"║  Improvement  : {improvement:+.1f}%"
      f"{'':<24} ║")
print("╠══════════════════════════════════════════╣")
if achieved:
    print("║  ✅ DQN beats all baselines!             ║")
else:
    print("║  ⚠️  Need more training episodes         ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → DQN Analysis + 1000 Seasons ║")
print("╚══════════════════════════════════════════╝")